In [3]:
from torch import nn
import torch
import pandas as pd
from gensim.models import KeyedVectors
import gensim.downloader as api
from torch.nn.utils.rnn import pad_sequence
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import numpy as np

In [2]:
# !pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 16.3 MB/s eta 0:00:00


In [5]:
df = pd.read_csv('/content/sentimentdataset.csv')

In [6]:
df.head()

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [7]:
df = df.set_index('Unnamed: 0')

In [8]:
X = df['Text']
Y= df['Sentiment']

In [9]:
glove = api.load("glove-wiki-gigaword-300")  # ~376MB

[==================================================] 100.0% 376.1/376.1MB downloaded


In [10]:
def tokenize(text):
    return text.lower().split()

In [11]:
def sentence_to_word_embeddings(sentence, model):
    tokens = tokenize(sentence)
    return [
        model[word]
        for word in tokens
        if word in model
    ]

In [12]:
X_embeddings = X.apply(lambda x: sentence_to_word_embeddings(x, glove))

In [13]:
# X_embeddings_tensors = torch.tensor( X_embeddings, dtype=torch.float32 )

X_tensors = [ torch.tensor(emb, dtype=torch.float32)   for emb in X_embeddings  ]

/tmp/ipykernel_4991/2135117197.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X_tensors = [ torch.tensor(emb, dtype=torch.float32)   for emb in X_embeddings  ]


In [14]:
X_padded = pad_sequence( X_tensors, batch_first=True, padding_value=0.0 )

In [15]:
print(X_padded.shape)

torch.Size([732, 22, 300])


In [16]:
lengths = torch.tensor([t.shape[0] for t in X_tensors])

In [36]:

class SentimentBiLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=512,
            num_layers=4,
            batch_first=True,
            dropout=0.3,
            bidirectional=True,
            proj_size=128
        )

        # Because bidirectional + proj_size=128
        self.classifier = nn.Linear(2 * 128, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        """
        x: (batch_size, max_seq_len, 300)
        lengths: (batch_size,)
        """

        # 1️⃣ Pack padded sequences
        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 2️⃣ LSTM forward
        packed_out, (h_n, c_n) = self.lstm(packed_x)

        # 3️⃣ Unpack
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        # out: (batch_size, max_seq_len, 256)

        # 4️⃣ Extract last valid timestep per sequence
        batch_size = out.size(0)
        last_outputs = out[
            torch.arange(batch_size),
            lengths - 1
        ]  # (batch_size, 256)

        # 5️⃣ Classification
        logits = self.classifier(self.dropout(last_outputs))

        return logits

In [48]:

class SentimentGRU(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.lstm = nn.GRU(
            input_size=300,
            hidden_size=128,
            num_layers=4,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Because bidirectional + proj_size=128
        self.classifier = nn.Linear(2 * 128, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        """
        x: (batch_size, max_seq_len, 300)
        lengths: (batch_size,)
        """

        # 1️⃣ Pack padded sequences
        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 2️⃣ LSTM forward
        packed_out, h_n = self.lstm(packed_x)

        # 3️⃣ Unpack
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        # out: (batch_size, max_seq_len, 256)

        # 4️⃣ Extract last valid timestep per sequence
        batch_size = out.size(0)
        last_outputs = out[
            torch.arange(batch_size),
            lengths - 1
        ]  # (batch_size, 256)

        # 5️⃣ Classification
        logits = self.classifier(self.dropout(last_outputs))

        return logits

In [49]:
help(nn.LSTM.__init__)

Help on function __init__ in module torch.nn.modules.rnn:

__init__(self, *args, **kwargs)
    Initialize internal Module state, shared by both nn.Module and ScriptModule.



In [50]:

num_samples = X_padded.size(0)
indices = torch.randperm(num_samples)

split = int(0.8 * num_samples)
train_idx, test_idx = indices[:split], indices[split:]

X_train = X_padded[train_idx]
X_test = X_padded[test_idx]

lengths_train = lengths[train_idx]
lengths_test = lengths[test_idx]


In [51]:
Y_enc = Y.apply(lambda x: 1 if str(x).strip().lower() =='positive' else 0)
Y_tensor = torch.tensor(Y_enc.values, dtype=torch.long)

In [52]:
y_train = Y_tensor[train_idx]
y_test = Y_tensor[test_idx]

In [53]:
X_train.shape

torch.Size([585, 22, 300])

In [54]:
y_train.shape

torch.Size([585])

In [55]:
len_tensor = torch.tensor(lengths, dtype=torch.long)

/tmp/ipykernel_4991/1649207563.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  len_tensor = torch.tensor(lengths, dtype=torch.long)


In [56]:
len_tensor_train = len_tensor[train_idx]
len_tensor_test = len_tensor[test_idx]

In [57]:
# epoch = 50
# model = SentimentBiLSTM()
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-3,
#     weight_decay=1e-2
# )


# for e in range(epoch):
#   loss_epoch =0
#   optimizer.zero_grad()

#   logits = model(X_train[1,:,:], len_tensor_train[i])

#   loss = criterion(logits, y_train[i])
#   loss.backward()
#   optimizer.step()
#   loss_epoch += loss

#   print(f'Loss : {loss} in epoch: {e}')

In [58]:
epoch = 50
model = SentimentGRU()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

model.train()

for e in range(epoch):
    optimizer.zero_grad()

    logits = model(X_train, len_tensor_train)     # ✅ batch forward
    loss = criterion(logits, y_train)    # ✅ batch labels

    loss.backward()
    optimizer.step()

    loss_epoch = loss.item()
    print(f"Epoch {e+1}/{epoch} | Loss: {loss_epoch:.4f}")

Epoch 1/50 | Loss: 0.7343
Epoch 2/50 | Loss: 0.5185
Epoch 3/50 | Loss: 0.3465
Epoch 4/50 | Loss: 0.2197
Epoch 5/50 | Loss: 0.1752
Epoch 6/50 | Loss: 0.1892
Epoch 7/50 | Loss: 0.2095
Epoch 8/50 | Loss: 0.2069
Epoch 9/50 | Loss: 0.2140
Epoch 10/50 | Loss: 0.2033
Epoch 11/50 | Loss: 0.1916
Epoch 12/50 | Loss: 0.1816
Epoch 13/50 | Loss: 0.1698
Epoch 14/50 | Loss: 0.1682
Epoch 15/50 | Loss: 0.1724
Epoch 16/50 | Loss: 0.1757
Epoch 17/50 | Loss: 0.1730
Epoch 18/50 | Loss: 0.1600
Epoch 19/50 | Loss: 0.1489
Epoch 20/50 | Loss: 0.1446
Epoch 21/50 | Loss: 0.1325
Epoch 22/50 | Loss: 0.1240
Epoch 23/50 | Loss: 0.1088
Epoch 24/50 | Loss: 0.1056
Epoch 25/50 | Loss: 0.0962
Epoch 26/50 | Loss: 0.0832
Epoch 27/50 | Loss: 0.0724
Epoch 28/50 | Loss: 0.0701
Epoch 29/50 | Loss: 0.0612
Epoch 30/50 | Loss: 0.0545
Epoch 31/50 | Loss: 0.0464
Epoch 32/50 | Loss: 0.0450
Epoch 33/50 | Loss: 0.0330
Epoch 34/50 | Loss: 0.0335
Epoch 35/50 | Loss: 0.0316
Epoch 36/50 | Loss: 0.0273
Epoch 37/50 | Loss: 0.0207
Epoch 38/5

In [59]:
model.eval()

SentimentGRU(
  (lstm): GRU(300, 128, num_layers=4, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [60]:
logits = model(X_test, len_tensor_test)

In [61]:
logits.shape

torch.Size([147, 2])

In [62]:
model.eval()
eval_loss = 0.0

with torch.no_grad():
  logits = model(X_test, len_tensor_test)
  loss = criterion(logits, y_test)

  print(f'Loss: {loss.item()}')

Loss: 0.4532921314239502


In [63]:
model.eval()

pred = []

with torch.no_grad():
    logits = model(X_test, len_tensor_test)
    pred = torch.argmax(logits, dim=1)

print(pred)

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0])


In [64]:
results= pd.DataFrame(np.array(pred), columns=['pred'])

In [65]:
results['test'] = pd.Series(np.array(y_test))

In [66]:

from sklearn.metrics import confusion_matrix

# Ensure on CPU + numpy
y_true = y_test.detach().cpu().numpy()
y_pred = pred.detach().cpu().numpy()

cm = confusion_matrix(y_true, y_pred)
cm

array([[129,   3],
       [  9,   6]])

In [67]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class MHAPooling(nn.Module):
    """
    Multi-head attention pooling with a learnable query vector
    """
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        # Learnable global query (1 per head implicitly)
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x, lengths):
        """
        x: (B, T, D)
        lengths: (B,)
        """

        B, T, D = x.size()

        # Expand query for batch
        query = self.query.expand(B, -1, -1)  # (B, 1, D)

        # Padding mask: True = ignore
        key_padding_mask = (
            torch.arange(T, device=lengths.device)
            .unsqueeze(0) >= lengths.unsqueeze(1)
        )  # (B, T)

        # Multi-head attention
        pooled, attn_weights = self.mha(
            query=query,           # (B, 1, D)
            key=x,                 # (B, T, D)
            value=x,               # (B, T, D)
            key_padding_mask=key_padding_mask
        )

        # pooled: (B, 1, D)
        pooled = pooled.squeeze(1)  # (B, D)

        return pooled, attn_weights


In [69]:

class MHAEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask):
        # Self‑Attention
        attn_out, attn_weights = self.mha(
            x, x, x,
            key_padding_mask=padding_mask,
            need_weights=True,
            average_attn_weights=False
        )

        x = self.norm1(x + self.dropout(attn_out))

        # Feed Forward
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x, attn_weights


In [68]:
class SentimentTransformerMHA(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        ff_dim=256,
        num_layers=2,
        num_classes=2,
        max_len=512,
        padding_idx=0,
        dropout=0.2,
        pretrained_embeddings=None
    ):
        super().__init__()

        # 1️⃣ Token & Positional Embeddings
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=padding_idx
        )

        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)

        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        # 2️⃣ Encoder Stack
        self.layers = nn.ModuleList([
            MHAEncoderLayer(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # 3️⃣ Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, input_ids, lengths):
        """
        input_ids: [B, T]
        lengths:   [B]
        """

        B, T = input_ids.shape

        # Padding mask (True = ignore position)
        padding_mask = torch.arange(T, device=lengths.device)\
            .expand(B, T) >= lengths.unsqueeze(1)

        # 1️⃣ Embeddings
        token_emb = self.embedding(input_ids)

        positions = torch.arange(T, device=input_ids.device)\
            .unsqueeze(0).expand(B, T)
        pos_emb = self.pos_embedding(positions)

        x = token_emb + pos_emb

        all_attn_weights = []

        # 2️⃣ Encoder Layers
        for layer in self.layers:
            x, attn_weights = layer(x, padding_mask)
            all_attn_weights.append(attn_weights)

        # Stack attention weights
        # [num_layers, B, num_heads, T, T]
        all_attn_weights = torch.stack(all_attn_weights)

        # 3️⃣ Attention‑aware Pooling (mask‑aware mean)
        mask = (~padding_mask).unsqueeze(-1)
        x = x * mask
        pooled = x.sum(dim=1) / mask.sum(dim=1)

        # 4️⃣ Classifier
        logits = self.classifier(pooled)

        return logits, all_attn_weights